# SHIPP Stage 8 — Gold candidate matches (scoring v1)

**Current Gold contract:** `v1-structured`.

Gold v1 uses trusted structured marketplace and route features only.
Semantic retrieval is handled separately by the AI Search / RAG path and
is not required by this validated Gold v1 scoring contract.

**Run:** Clear state and outputs → Run all.
Every gate prints `PASS` or stops with an assertion.
Never loosen an assertion to make a failed dataset pass.

This notebook is the only writer of `gold_candidate_matches`.

In [0]:
%run ../common/shipp_silver_lib

In [0]:
from data_pipeline.gold import scoring

# -------------------------------------------------------------------
# 1. Validate scoring configuration and Gold schema
# -------------------------------------------------------------------

scoring.assert_weights_valid()

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}"
)


In [0]:
# -------------------------------------------------------------------
# 2. Final Gold contract
#
# Keep the existing analytics/business columns AND add the Agent-facing
# compatibility columns required by shipp.agent.contracts.
# -------------------------------------------------------------------

EXPECTED_COLUMNS = [
    # Business keys / ranking
    "request_id",
    "listing_id",
    "match_rank",
    "match_score",
    "match_pct",

    # Score components
    "distance_score",
    "timing_score",
    "condition_score",

    # Routing
    "route_status",
    "route_distance_km",
    "route_duration_min",
    "straight_line_km",

    # Listing / request context
    "category",
    "listing_title",
    "listing_condition",
    "listing_status",
    "available_from",
    "available_until",
    "need_by_date",

    # Gold metadata
    "scoring_version",
    "gold_processed_at",

    # ---------------------------------------------------------------
    # Agent-facing compatibility contract
    # Required by CANDIDATE_MATCH_COLUMNS
    # ---------------------------------------------------------------
    "title",
    "condition",
    "area",
    "distance_km",
    "duration_min",
    "computed_at",
]



In [0]:
# -------------------------------------------------------------------
# 3. Read trusted Silver inputs
# -------------------------------------------------------------------

# Fail clearly if an upstream Stage 5–7 product is missing.
for required_table in [
    SILVER_CANDIDATE_PAIRS,
    SILVER_ROUTES,
    SILVER_LISTINGS,
    SILVER_REQUESTS,
]:
    assert table_exists(required_table), (
        f"BLOCKED: missing required upstream table {required_table}"
    )

print("PASS — all Gold upstream dependencies exist.")


# Candidate pairs
pairs = (
    spark.table(SILVER_CANDIDATE_PAIRS)
    .alias("p")
)


# Trusted normalized routes
routes = (
    spark.table(SILVER_ROUTES)
    .alias("rt")
)


# Current eligible listings
listings = (
    spark.table(SILVER_LISTINGS)
    .filter(
        F.col("status").isin(
            ELIGIBLE_LISTING_STATUSES
        )
    )
    .alias("l")
)


# Current eligible requests
requests_df = (
    spark.table(SILVER_REQUESTS)
    .filter(
        F.col("status").isin(
            ELIGIBLE_REQUEST_STATUSES
        )
    )
    .alias("r")
)


now = now_ntz()


print(
    "Candidate pairs rows:",
    pairs.count(),
)

print(
    "Silver routes rows:",
    routes.count(),
)

print(
    "Eligible listings rows:",
    listings.count(),
)

print(
    "Eligible requests rows:",
    requests_df.count(),
)

In [0]:

# -------------------------------------------------------------------
# 4. Join candidate pairs + routes + trusted current Silver data
# -------------------------------------------------------------------

joined = (
    pairs
    .join(
        routes,
        ["origin_key", "dest_key", "profile"],
        "left",
    )
    .join(
        listings,
        F.col("p.listing_id") == F.col("l.listing_id"),
        "inner",
    )
    .join(
        requests_df,
        F.col("p.request_id") == F.col("r.request_id"),
        "inner",
    )
    .select(
        F.col("p.request_id").alias("request_id"),
        F.col("p.listing_id").alias("listing_id"),

        F.coalesce(
            F.col("rt.route_status"),
            F.lit("PENDING"),
        ).alias("route_status"),

        F.when(
            F.col("rt.route_status") == "OK",
            F.col("rt.distance_km"),
        ).alias("route_distance_km"),

        F.when(
            F.col("rt.route_status") == "OK",
            F.col("rt.duration_min"),
        ).alias("route_duration_min"),

        F.col("p.straight_line_km"),
        F.col("p.category"),

        F.col("l.title").alias("listing_title"),
        F.col("l.condition").alias("listing_condition"),
        F.col("l.status").alias("listing_status"),

        F.col("l.available_from"),
        F.col("l.available_until"),

        F.col("r.need_by_date"),
    )
)




In [0]:
# -------------------------------------------------------------------
# 5. Calculate structured match scores
# -------------------------------------------------------------------

scored = (
    joined
    .withColumn(
        "distance_score",
        F.round(
            scoring.distance_score(
                F.col("route_status"),
                F.col("route_distance_km"),
            ),
            4,
        ),
    )
    .withColumn(
        "timing_score",
        F.round(
            scoring.timing_score(
                F.col("available_from"),
                F.col("need_by_date"),
                now,
            ),
            4,
        ),
    )
    .withColumn(
        "condition_score",
        scoring.condition_score(
            F.col("listing_condition")
        ),
    )
    .withColumn(
        "match_score",
        scoring.match_score(
            F.col("distance_score"),
            F.col("timing_score"),
            F.col("condition_score"),
        ),
    )
)




In [0]:
# -------------------------------------------------------------------
# 6. Rank matches per request
# -------------------------------------------------------------------

rank = (
    Window
    .partitionBy("request_id")
    .orderBy(
        F.col("match_score").desc(),
        F.col("route_distance_km").asc_nulls_last(),
        F.col("listing_id"),
    )
)




In [0]:
# -------------------------------------------------------------------
# 7. Build Gold + Agent-facing compatibility columns
# -------------------------------------------------------------------

gold_df = (
    scored
    .withColumn(
        "match_rank",
        F.row_number().over(rank),
    )
    .filter(
        F.col("match_rank")
        <= SCORING["top_n_per_request"]
    )
    .withColumn(
        "match_pct",
        F.round(
            F.col("match_score") * 100
        ).cast("int"),
    )
    .withColumn(
        "scoring_version",
        F.lit(SCORING["version"]),
    )
    .withColumn(
        "gold_processed_at",
        F.current_timestamp(),
    )

    # ===============================================================
    # Agent contract aliases
    # ===============================================================

    .withColumn(
        "title",
        F.col("listing_title"),
    )
    .withColumn(
        "condition",
        F.col("listing_condition"),
    )

    # Current Silver contract does not expose a trusted coarse area.
    # Agent contract permits area=None, so do NOT fabricate location.
    .withColumn(
        "area",
        F.lit(None).cast("string"),
    )

    .withColumn(
        "distance_km",
        F.col("route_distance_km"),
    )
    .withColumn(
        "duration_min",
        F.col("route_duration_min"),
    )
    .withColumn(
        "computed_at",
        F.col("gold_processed_at"),
    )

    .select(*EXPECTED_COLUMNS)
)


# -------------------------------------------------------------------
# 8. Runtime evidence
# -------------------------------------------------------------------

row_count = gold_df.count()

print("Gold candidate rows:", row_count)

display(
    gold_df
    .select(
        "request_id",
        "match_rank",
        "listing_id",
        "title",
        "match_pct",
        "route_status",
        "distance_km",
        "duration_min",
        "area",
        "computed_at",
    )
    .orderBy(
        "request_id",
        "match_rank",
    )
)

In [0]:
## PENDING check

pending = gold_df.filter(
    F.col("route_status") == "PENDING"
).count()

print("Pending Gold routes:", pending)

## Quality gate — the Agent is blocked until this passes

In [0]:
# -------------------------------------------------------------------
# 9. Final Gold quality gate
# -------------------------------------------------------------------

run_quality_gate(
    {
        # -----------------------------------------------------------
        # Business-key / ranking integrity
        # -----------------------------------------------------------
        "duplicate (request, listing)": duplicate_key_count(
            gold_df,
            "request_id",
            "listing_id",
        ),

        "duplicate (request, rank)": duplicate_key_count(
            gold_df,
            "request_id",
            "match_rank",
        ),

        "null request_id": gold_df.filter(
            F.col("request_id").isNull()
        ).count(),

        "null listing_id": gold_df.filter(
            F.col("listing_id").isNull()
        ).count(),

        # -----------------------------------------------------------
        # Score integrity
        # -----------------------------------------------------------
        "null match_score": gold_df.filter(
            F.col("match_score").isNull()
        ).count(),

        "score outside [0,1]": gold_df.filter(
            F.col("match_score").isNotNull()
            & ~F.col("match_score").between(0, 1)
        ).count(),

        "match_pct outside [0,100]": gold_df.filter(
            F.col("match_pct").isNotNull()
            & ~F.col("match_pct").between(0, 100)
        ).count(),

        # -----------------------------------------------------------
        # Eligibility
        # -----------------------------------------------------------
        "listing not eligible": gold_df.filter(
            F.col("listing_status").isNull()
            | ~F.col("listing_status").isin(
                ELIGIBLE_LISTING_STATUSES
            )
        ).count(),

        # -----------------------------------------------------------
        # Route contract
        # -----------------------------------------------------------
        "invalid route status": gold_df.filter(
            ~F.col("route_status").isin(
                "OK",
                "NO_ROUTE",
                "PENDING",
            )
        ).count(),

        "PENDING route": pending,

        "distance without OK route": gold_df.filter(
            (F.col("route_status") != "OK")
            & F.col("route_distance_km").isNotNull()
        ).count(),

        "duration without OK route": gold_df.filter(
            (F.col("route_status") != "OK")
            & F.col("route_duration_min").isNotNull()
        ).count(),

        "OK route without distance": gold_df.filter(
            (F.col("route_status") == "OK")
            & F.col("route_distance_km").isNull()
        ).count(),

        "OK route without duration": gold_df.filter(
            (F.col("route_status") == "OK")
            & F.col("route_duration_min").isNull()
        ).count(),
    },
    row_count=row_count,
    allow_empty=True,
)


# -------------------------------------------------------------------
# 10. Persist Gold
# -------------------------------------------------------------------

written = write_overwrite(
    gold_df,
    GOLD_CANDIDATE_MATCHES,
)

assert written == row_count, (
    f"WRITE VALIDATION FAILED: "
    f"wrote {written}, expected {row_count}"
)

assert_exact_schema(
    GOLD_CANDIDATE_MATCHES,
    EXPECTED_COLUMNS,
)


# -------------------------------------------------------------------
# 11. Persisted-table validation
# -------------------------------------------------------------------

persisted = spark.table(
    GOLD_CANDIDATE_MATCHES
)

persisted_row_count = persisted.count()

assert persisted_row_count == written, (
    "PERSISTENCE VALIDATION FAILED: "
    f"expected {written} rows, "
    f"found {persisted_row_count}"
)

persisted_duplicate_pairs = duplicate_key_count(
    persisted,
    "request_id",
    "listing_id",
)

assert persisted_duplicate_pairs == 0, (
    "PERSISTENCE VALIDATION FAILED: "
    f"{persisted_duplicate_pairs} duplicate "
    "(request_id, listing_id) rows found"
)


requests_with_matches = (
    persisted
    .select("request_id")
    .distinct()
    .count()
)


print(
    "PASS — persisted Gold candidate matches validated."
)


# -------------------------------------------------------------------
# 12. Final summary
# -------------------------------------------------------------------

summary = {
    "table": GOLD_CANDIDATE_MATCHES,
    "rows": written,
    "pending_routes": pending,
    "requests_with_matches": requests_with_matches,
    "status": "PASS",
}

print(summary)

dbutils.notebook.exit(
    json.dumps(summary)
)